## 1. 环境准备

导入必要的库并配置中文字体支持。

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib import rcParams

# ================== 全局环境配置 ==================
# 配置 matplotlib 使用中文字体
# 注意：run_jupyterlab.sh 已通过挂载 matplotlibrc 文件进行全局配置
# 这里再次配置是为了确保在非 Docker 环境（如本地直接运行）下也能正确显示中文
def config_fonts():
    try:
        # 尝试多种常用中文字体，优先使用挂载的字体
        # SimHei: Windows标准/Docker挂载
        # Heiti TC: Mac标准
        fonts = ['SimHei', 'Heiti TC', 'WenQuanYi Micro Hei', 'DejaVu Sans']
        rcParams['font.sans-serif'] = fonts
        rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
    except:
        pass

config_fonts()
print("环境配置完成，中文字体已设置。")


环境配置完成，中文字体已设置。


## 2. 朴素贝叶斯模型构建与评估

本部分演示高斯朴素贝叶斯 (GaussianNB) 算法在鸢尾花分类问题中的应用：
1.  **数据加载**：加载鸢尾花数据集。
2.  **数据预处理**：按 7:3 比例划分训练集和测试集。
3.  **模型构建**：使用 `GaussianNB`，该模型假设特征符合高斯分布，适用于连续数值特征。
4.  **模型训练与评估**：拟合训练数据，并在测试集上计算准确率、生成分类报告和混淆矩阵。

In [14]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 加载鸢尾花数据集
iris = load_iris()
X = iris.data  # 特征矩阵
y = iris.target  # 目标标签

# 将数据集划分为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 创建高斯朴素贝叶斯分类器
gnb = GaussianNB()

# 使用训练数据训练模型
gnb.fit(X_train, y_train)

# 使用测试数据进行预测
y_pred = gnb.predict(X_test)

# 计算预测准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"模型准确率: {accuracy:.2f}")

# 输出分类报告
print("分类报告:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 输出混淆矩阵
print("混淆矩阵:")
print(confusion_matrix(y_test, y_pred))

模型准确率: 0.98
分类报告:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      0.92      0.96        13
   virginica       0.93      1.00      0.96        13

    accuracy                           0.98        45
   macro avg       0.98      0.97      0.97        45
weighted avg       0.98      0.98      0.98        45

混淆矩阵:
[[19  0  0]
 [ 0 12  1]
 [ 0  0 13]]


## 3. 进阶：文本分类实战 (MultinomialNB)

朴素贝叶斯在文本分类（如垃圾邮件过滤、新闻分类）中应用广泛。
对于文本数据，特征通常是词频（出现的次数），因此更适合使用多项式朴素贝叶斯 (`MultinomialNB`)。

本节演示如何使用 `CountVectorizer` 将文本转换为特征向量，并使用 `MultinomialNB` 进行分类。

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# 1. 构造简单的文本数据集 (模拟垃圾邮件分类)
# 样本数据
train_docs = [
    'offer is secret',          # spam
    'click secret link',        # spam
    'secret sports link',       # spam
    'play sports',              # ham (not spam)
    'play baseball',            # ham
    'sports is healthy'         # ham
]
y_train = [1, 1, 1, 0, 0, 0]  # 1=spam, 0=ham

test_docs = [
    'sports is fun',            # ham
    'secret offer',             # spam
    'will play baseball',       # ham
    'click this offer'          # spam
]
y_test = [0, 1, 0, 1]

# 2. 特征提取 (词频统计)
vec = CountVectorizer()
X_train_dtm = vec.fit_transform(train_docs)
X_test_dtm = vec.transform(test_docs)

print("词汇表:", vec.get_feature_names_out())
print("训练集特征矩阵 (Dense):\n", X_train_dtm.toarray())

# 3. 训练 MultinomialNB 模型
# alpha=1.0 默认启用拉普拉斯平滑
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train_dtm, y_train)

# 4. 预测
y_pred_class = mnb.predict(X_test_dtm)
y_pred_prob = mnb.predict_proba(X_test_dtm)

print("\n测试集预测结果:", y_pred_class)
print("真实标签:      ", y_test)
print("预测准确率:", accuracy_score(y_test, y_pred_class))

词汇表: ['baseball' 'click' 'healthy' 'is' 'link' 'offer' 'play' 'secret' 'sports']
训练集特征矩阵 (Dense):
 [[0 0 0 1 0 1 0 1 0]
 [0 1 0 0 1 0 0 1 0]
 [0 0 0 0 1 0 0 1 1]
 [0 0 0 0 0 0 1 0 1]
 [1 0 0 0 0 0 1 0 0]
 [0 0 1 1 0 0 0 0 1]]

测试集预测结果: [0 1 0 1]
真实标签:       [0, 1, 0, 1]
预测准确率: 1.0


## 4. 进阶：伯努利朴素贝叶斯 (BernoulliNB)

如果特征是二值的（即词语是否出现，而不考虑出现次数），或者我们明确将特征二值化，则可以使用伯努利朴素贝叶斯 (`BernoulliNB`)。
它对于短文本（如标题、短信）分类有时效果更好。

In [16]:
from sklearn.naive_bayes import BernoulliNB

# 1. 使用 binary=True 将词频向量转换为二值向量 (0/1)
# 或者在 CountVectorizer 中设置 binary=True
vec_binary = CountVectorizer(binary=True)
X_train_bin = vec_binary.fit_transform(train_docs)
X_test_bin = vec_binary.transform(test_docs)

print("二值化特征矩阵 (Train):\n", X_train_bin.toarray())

# 2. 训练 BernoulliNB 模型
bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_train_bin, y_train)

# 3. 预测
y_pred_bnb = bnb.predict(X_test_bin)
print("\nBernoulliNB 预测结果:", y_pred_bnb)
print("真实标签:            ", y_test)
print("预测准确率:", accuracy_score(y_test, y_pred_bnb))

二值化特征矩阵 (Train):
 [[0 0 0 1 0 1 0 1 0]
 [0 1 0 0 1 0 0 1 0]
 [0 0 0 0 1 0 0 1 1]
 [0 0 0 0 0 0 1 0 1]
 [1 0 0 0 0 0 1 0 0]
 [0 0 1 1 0 0 0 0 1]]

BernoulliNB 预测结果: [0 1 0 1]
真实标签:             [0, 1, 0, 1]
预测准确率: 1.0


## 5. 进阶：拉普拉斯平滑 (Laplace Smoothing)

在计算条件概率时，如果某个特征值在训练集中没有出现过（频数为0），会导致概率为0，从而影响整体预测（零概率问题）。
拉普拉斯平滑通过在分子分母中加入平滑系数 $\alpha$ 来解决这个问题。

- `alpha=1.0`: 默认值，拉普拉斯平滑。
- `alpha=0.0`: 不平滑（可能导致错误）。
- `alpha < 1.0`: Lidstone 平滑。

In [17]:
# 演示平滑的作用
# 构造一个测试样本，包含训练集中未出现的词 (这里 CountVectorizer 会忽略未登录词，
# 但如果是自己实现的贝叶斯或者某些特定情况，未登录特征会导致问题。
# 在 sklearn 中，CountVectorizer 词表是固定的。如果测试集有新词，会被忽略。
# 这里主要演示 alpha 对概率估计的影响。)

# 我们手动构造一个计数矩阵，模拟有一个特征在某类中计数为0
X_demo = np.array([[1, 0], [1, 0], [1, 0], [0, 1]]) # 3个类0，1个类1
y_demo = np.array([0, 0, 0, 1])
# 特征2在类0中出现了0次

# 不使用平滑 (alpha=1e-10 近似 0, sklearn 不允许完全为0)
clf_no_smooth = MultinomialNB(alpha=1e-10)
clf_no_smooth.fit(X_demo, y_demo)

# 使用平滑 (alpha=1)
clf_smooth = MultinomialNB(alpha=1.0)
clf_smooth.fit(X_demo, y_demo)

# 查看特征对数概率 (log probability)
print("不平滑时的特征对数概率 (类0):", clf_no_smooth.feature_log_prob_[0])
print("平滑后的特征对数概率 (类0):  ", clf_smooth.feature_log_prob_[0])

# 解释：不平滑时，计数为0的特征概率极低 (log后为负无穷大，这里受限于浮点精度和1e-10)
# 平滑后，概率被拉离0，避免了‘一票否决’。 


不平滑时的特征对数概率 (类0): [-3.33333361e-11 -2.41244632e+01]
平滑后的特征对数概率 (类0):   [-0.22314355 -1.60943791]
